# Notebook 05 — Unified Ingest (per Institution / Department)
### Rancang Bangun Chatbot Sumber Informasi Sivitas UPI Berbasis RAG

One place to ingest **any** source into the shared FAISS index — UPI Cibiru, UPI
Sumedang, and document folders such as PMB UPI, Dit-Pendidikan, BiroSDM, PPID, LPPM,
Kalender Akademik, or `New_Dataset`.

**Design**
- A **source registry** (Cell 2): add a new institution/department by appending one entry.
- Two source kinds:
  - `web_md`  — already-scraped markdown (`<folder>/text/*.md` + optional `manifest.csv`).
  - `raw_docs` — raw files (`.pdf .html .txt .docx .xlsx`) → extract → clean → chunk.
- **Append-only & safe**: de-dupes by `doc_id` (already indexed) **and** by exact chunk
  text (content overlap), backs up the index before writing, keeps FAISS row *i* ↔
  `meta[i]` aligned, and uses the same embedding model the index was built with.
- **GPU auto**: uses CUDA if available (`CONFIG["device"]="auto"`), else CPU.

**How to use**: Run Cell 1→4 (setup). Run **Cell 5** to see what's already indexed.
Set `KEY` in **Cell 6** and dry-run. Then set `CONFIRM=True` in **Cell 7** to embed+append.

> Requires the base index to exist (`_pipeline/index/faiss.index` + `chunks_meta.json`).

## Cell 1 — Config, imports, helpers

In [ ]:
# =============================================================================
# Cell 1 — config / imports / helpers
# =============================================================================
import json, re, os, io, time, hashlib, logging, shutil
from pathlib import Path
from datetime import datetime
from collections import Counter
import numpy as np

BASE = Path(os.environ.get("RAG_UPI_BASE", r"D:\Project\RAG_UPI\Dataset"))
PIPE = BASE / "_pipeline"
INDEX_DIR    = PIPE / "index"
CHUNKS_JSONL = PIPE / "chunked" / "chunks.jsonl"
INDEX_FILE   = INDEX_DIR / "faiss.index"
META_FILE    = INDEX_DIR / "chunks_meta.json"
INFO_FILE    = INDEX_DIR / "index_info.json"
SHARD_DIR    = INDEX_DIR / "shards"
for d in (INDEX_DIR, CHUNKS_JSONL.parent, SHARD_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # Embedding — MUST match what built the existing FAISS index
    "embedding_model": "intfloat/multilingual-e5-base",
    "use_e5_prefixes": True,
    "embedding_batch_size": 64,
    "device": "auto",                 # "auto" -> cuda if available else cpu ; or "cpu"/"cuda"
    # Chunking
    "web_chunk_size": 900,  "web_chunk_overlap": 150,
    "doc_chunk_size": 1000, "doc_chunk_overlap": 200,
    "min_chunk_chars": 50,
    # Extras
    "keywords_top": 8,
    "skip_duplicate_text": True,      # content-level dedup (md5 of normalized text)
    # OCR (raw PDF fallback)
    "ocr_min_chars_per_page": 80, "ocr_dpi": 200, "ocr_languages_tesseract": "ind+eng",
}

def get_logger():
    lg = logging.getLogger("ingest05"); lg.setLevel(logging.INFO)
    if not lg.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-5s | %(message)s", "%H:%M:%S"))
        lg.addHandler(h)
    lg.propagate = False
    return lg
log = get_logger()

_WS = re.compile(r"\s+")
def clean_ws(t):   return _WS.sub(" ", (t or "").replace(" ", " ")).strip()
def sha16(s):      return hashlib.sha1(str(s).encode("utf-8")).hexdigest()[:16]
def text_hash(t):  return hashlib.md5(clean_ws(t).lower().encode("utf-8")).hexdigest()

def read_json(p, default=None):
    p = Path(p)
    if not p.exists(): return default
    try: return json.loads(p.read_text(encoding="utf-8"))
    except json.JSONDecodeError: return default

def write_json(p, obj):
    p = Path(p); tmp = p.with_suffix(p.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8"); tmp.replace(p)

def pick_device():
    d = CONFIG["device"]
    if d == "auto":
        try:
            import torch
            return "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            return "cpu"
    return d

log.info("BASE   = %s", BASE)
log.info("device = %s", pick_device())
log.info("index  = %s (%s)", INDEX_FILE, "exists" if INDEX_FILE.exists() else "MISSING")

## Cell 2 — Source registry  (add a new institution/department here)

In [ ]:
# =============================================================================
# Cell 2 — SOURCE REGISTRY
#   kind="web_md"   : folder of *.md (+ optional manifest.csv with source_url)
#   kind="raw_docs" : raw files (.pdf .html .txt .docx .xlsx) to extract+clean
#   doc_prefix      : namespace for doc_id so sources never collide
#   category        : label stored on every chunk (used for filtering/citation)
# To add a source: copy a line, point it at the folder, give it a unique doc_prefix.
# =============================================================================
SOURCES = {
  # ---- Web sites already scraped to markdown ----
  "upi_cibiru":   dict(kind="web_md", category="UPI_Cibiru_Web",   doc_prefix="cibiru",
                       md_dir=BASE/"UPI Cibiru"/"text",   manifest=BASE/"UPI Cibiru"/"manifest.csv"),
  "upi_sumedang": dict(kind="web_md", category="UPI_Sumedang_Web", doc_prefix="sumedang",
                       md_dir=BASE/"UPI Sumedang"/"text", manifest=BASE/"UPI Sumedang"/"manifest.csv"),
  # ---- TEMPLATE for a future web source (edit + uncomment) ----
  # "upi_main":   dict(kind="web_md", category="UPI_Main_Web",     doc_prefix="upimain",
  #                    md_dir=BASE/"UPI Main"/"text",   manifest=BASE/"UPI Main"/"manifest.csv"),

  # ---- Department / unit document folders (raw files) ----
  "pmb_upi":        dict(kind="raw_docs", category="PMB_UPI",          doc_prefix="pmb",     root=BASE/"PMB UPI"),
  "dit_pendidikan": dict(kind="raw_docs", category="Dit_Pendidikan",   doc_prefix="ditpend", root=BASE/"Dit-Pendidikan-UPI"),
  "biro_sdm":       dict(kind="raw_docs", category="BiroSDM",          doc_prefix="birosdm", root=BASE/"BiroSDM"),
  "ppid":           dict(kind="raw_docs", category="PPID",             doc_prefix="ppid",    root=BASE/"PPID"),
  "lppm":           dict(kind="raw_docs", category="LPPM_UPI",         doc_prefix="lppm",    root=BASE/"LPPM UPI"),
  "kalender":       dict(kind="raw_docs", category="Kalender_Akademik",doc_prefix="kalender",root=BASE/"Kalender Akademik"),
  # New_Dataset catch-all (mixed depts) — category taken from each subfolder name:
  "new_dataset":    dict(kind="raw_docs", category="New_Dataset",      doc_prefix="newds",   root=BASE/"New_Dataset", by_subfolder=True),
}
DOC_EXTS = {".pdf", ".html", ".htm", ".txt", ".docx", ".xlsx"}
log.info("registered sources: %s", list(SOURCES))

## Cell 3 — Loaders: discover (cheap) + build chunk-rows (web_md / raw_docs)

In [ ]:
# =============================================================================
# Cell 3 — discovery + chunk builders
#   discover_source(key) -> [{doc_id, path, base, category}]   (cheap; no extraction)
#   build_rows(key, items) -> [chunk-row dict]                 (reads/extracts only `items`)
# =============================================================================
import csv as _csv

# ---- text splitter (LangChain if available, else simple sliding window) ----
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    def split_text(text, size, overlap):
        sp = RecursiveCharacterTextSplitter(
            chunk_size=size, chunk_overlap=overlap,
            separators=["\n## ", "\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""],
            length_function=len, keep_separator=True)
        return [c.strip() for c in sp.split_text(text) if c.strip()]
except Exception:
    def split_text(text, size, overlap):
        text = text.strip(); out = []; i = 0; n = len(text)
        while i < n:
            end = min(i + size, n); out.append(text[i:end].strip())
            if end >= n: break
            i = end - overlap
        return [c for c in out if c]

# ---- TF-IDF keywords (display metadata) ----
ID_STOP = set("yang dan di ke dari untuk pada dengan atau adalah ini itu para sebagai dalam akan "
              "tidak juga oleh agar bagi serta dapat telah secara nomor tahun no the of and a an in on for to".split())
def add_keywords(rows, top):
    if not rows: return
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
    except Exception:
        for r in rows: r["keywords"] = []
        return
    vec = TfidfVectorizer(stop_words=list(ID_STOP), token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]{2,}\b",
                          max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
    X = vec.fit_transform(clean_ws(r["text"]) for r in rows); terms = vec.get_feature_names_out(); Xc = X.tocsr()
    for i, r in enumerate(rows):
        row = Xc.getrow(i)
        r["keywords"] = [terms[j] for j in row.indices[np.argsort(row.data)[::-1][:top]]] if row.nnz else []

# ---- web markdown parsing ----
def _md_parse(md):
    lines = md.split("\n"); title = (lines[0] if lines else "").lstrip("#").strip(); src = ""; bs = 0
    for i, ln in enumerate(lines[:6]):
        if ln.startswith("> Source:"):
            src = ln.replace("> Source:", "").strip(); bs = i + 1; break
    return title, src, "\n".join(lines[bs:]).strip()

def _load_manifest(path):
    m = {}
    if path and Path(path).exists():
        with open(path, encoding="utf-8-sig", newline="") as f:
            for row in _csv.DictReader(f): m[row.get("filename")] = row
    return m

# ---- raw extraction (lazy imports so web sources don't need these libs) ----
def _locate_tesseract():
    import pytesseract, shutil as _sh
    f = _sh.which("tesseract")
    if f: pytesseract.pytesseract.tesseract_cmd = f; return f
    for c in (r"C:\Program Files\Tesseract-OCR\tesseract.exe", r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"):
        if Path(c).exists(): pytesseract.pytesseract.tesseract_cmd = c; return c
    return None

def _extract_pdf(path):
    import fitz
    out = []; doc = fitz.open(path)
    try:
        for i, page in enumerate(doc):
            t = page.get_text("text").strip()
            if len(t) < CONFIG["ocr_min_chars_per_page"]:
                try:
                    from PIL import Image; import pytesseract
                    _locate_tesseract()
                    pix = page.get_pixmap(dpi=CONFIG["ocr_dpi"]); img = Image.open(io.BytesIO(pix.tobytes("png")))
                    o = pytesseract.image_to_string(img, lang=CONFIG["ocr_languages_tesseract"]).strip()
                    if len(o) > len(t): t = o
                except Exception: pass
            out.append((i + 1, t))
    finally:
        doc.close()
    return out

def _extract_html(path):
    raw = Path(path).read_text(encoding="utf-8", errors="ignore"); text = ""
    try:
        import trafilatura
        text = trafilatura.extract(raw, include_comments=False, include_tables=True, favor_recall=True) or ""
    except Exception: pass
    if len(text.strip()) < 100:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(raw, "lxml")
        for t in soup(["nav", "header", "footer", "script", "style", "aside", "form", "noscript"]): t.decompose()
        main = soup.find("main") or soup.find("article") or soup.body or soup
        text = main.get_text("\n") if main else ""
    return [(1, re.sub(r"\n{3,}", "\n\n", text).strip())]

def _extract_docx(path):
    import docx
    d = docx.Document(str(path)); parts = [p.text for p in d.paragraphs if p.text and p.text.strip()]
    for tbl in d.tables:
        for row in tbl.rows:
            cells = [clean_ws(c.text) for c in row.cells if clean_ws(c.text)]
            if cells: parts.append(" | ".join(cells))
    return [(1, "\n".join(parts))]

def _extract_xlsx(path):
    import pandas as pd
    sheets = pd.read_excel(path, sheet_name=None, header=None, dtype=str); out = []
    for name, df in sheets.items():
        out.append(f"## Sheet: {name}")
        for _, row in df.iterrows():
            cells = [clean_ws(str(c)) for c in row if str(c) != "nan" and clean_ws(str(c))]
            if cells: out.append(" | ".join(cells))
    return [(1, "\n".join(out))]

def extract_raw(path):
    ext = path.suffix.lower()
    if ext == ".pdf":            return _extract_pdf(path), "pdf"
    if ext in (".html", ".htm"): return _extract_html(path), "html"
    if ext == ".docx":           return _extract_docx(path), "docx"
    if ext == ".xlsx":           return _extract_xlsx(path), "xlsx"
    if ext == ".txt":            return [(1, path.read_text(encoding="utf-8", errors="ignore").strip())], "txt"
    return None, None

def discover_source(key):
    src = SOURCES[key]
    if src["kind"] == "web_md":
        md_dir = Path(src["md_dir"])
        return [dict(doc_id=sha16(src["doc_prefix"] + "/" + p.stem), path=p, base=p.stem, category=src["category"])
                for p in sorted(md_dir.glob("*.md"))]
    root = Path(src["root"]); items = []
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix.lower() not in DOC_EXTS or p.name.startswith("~$"): continue
        rel = p.relative_to(root)
        cat = src["category"]
        if src.get("by_subfolder"): cat = rel.parts[0] if len(rel.parts) > 1 else src["category"]
        items.append(dict(doc_id=sha16(src["doc_prefix"] + "/" + str(rel).lower()), path=p, base=p.stem, category=cat))
    return items

def build_rows(key, items):
    src = SOURCES[key]; rows = []
    if src["kind"] == "web_md":
        man = _load_manifest(src.get("manifest")); size, ov = CONFIG["web_chunk_size"], CONFIG["web_chunk_overlap"]
        for it in items:
            title, srcurl, body = _md_parse(it["path"].read_text(encoding="utf-8"))
            info = man.get(it["base"], {}); url = info.get("source_url") or srcurl or None; title = info.get("title") or title
            for ci, piece in enumerate(split_text(clean_ws(body), size, ov)):
                if len(piece) < CONFIG["min_chunk_chars"]: continue
                rows.append(dict(doc_id=it["doc_id"], source=str(it["path"]), url=url, title=title,
                                 category=it["category"], source_type="md", page=1, section=None,
                                 text=piece, chunk_length=len(piece), chunk_index=ci, chunk_id=f"{it['doc_id']}::{ci}"))
        return rows
    # raw_docs
    size, ov = CONFIG["doc_chunk_size"], CONFIG["doc_chunk_overlap"]
    for n, it in enumerate(items, 1):
        try:
            pages, st = extract_raw(it["path"])
        except Exception as e:
            log.warning("extract failed %s: %s", it["path"].name, e); continue
        if not pages: continue
        for pageno, ptext in pages:
            ptext = clean_ws(ptext)
            if len(ptext) < CONFIG["min_chunk_chars"]: continue
            for ci, piece in enumerate(split_text(ptext, size, ov)):
                if len(piece) < CONFIG["min_chunk_chars"]: continue
                rows.append(dict(doc_id=it["doc_id"], source=str(it["path"]), url=None, title=it["base"],
                                 category=it["category"], source_type=st, page=pageno, section=None,
                                 text=piece, chunk_length=len(piece), chunk_index=ci,
                                 chunk_id=f"{it['doc_id']}::{pageno}:{ci}"))
        if n % 25 == 0: log.info("  extracted %d/%d files ...", n, len(items))
    return rows

## Cell 4 — Embed + append (dedup, backup, alignment checks) + runner

In [ ]:
# =============================================================================
# Cell 4 — ingest_source(key, dry_run=True) / ingest_all(...)
# =============================================================================
_model = None
def get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        dev = pick_device(); log.info("loading %s on %s ...", CONFIG["embedding_model"], dev)
        _model = SentenceTransformer(CONFIG["embedding_model"], device=dev)
    return _model

def embed_passages(texts):
    m = get_model()
    pre = ["passage: " + t for t in texts] if CONFIG["use_e5_prefixes"] else list(texts)
    return m.encode(pre, batch_size=CONFIG["embedding_batch_size"], convert_to_numpy=True,
                    normalize_embeddings=True, show_progress_bar=len(texts) > 64).astype("float32")

def _existing_index_state():
    meta = read_json(META_FILE, default=[])
    known_docs = {r.get("doc_id") for r in meta}
    known_txt = {text_hash(r.get("text", "")) for r in meta} if CONFIG["skip_duplicate_text"] else set()
    return meta, known_docs, known_txt

def ingest_source(key, dry_run=True):
    import faiss
    assert key in SOURCES, f"unknown source '{key}'. Known: {list(SOURCES)}"
    t0 = time.time(); src = SOURCES[key]
    log.info("=== INGEST '%s'  (kind=%s, category=%s) ===", key, src["kind"], src["category"])

    items = discover_source(key)
    meta, known_docs, known_txt = _existing_index_state()
    new_items = [it for it in items if it["doc_id"] not in known_docs]
    log.info("files found=%d | already indexed=%d | new files=%d",
             len(items), len(items) - len(new_items), len(new_items))
    if not new_items:
        log.info("nothing new for '%s' (all files already in index).", key); return 0

    rows = build_rows(key, new_items)
    log.info("chunks built from new files: %d", len(rows))

    if CONFIG["skip_duplicate_text"]:
        seen = set(known_txt); kept = []
        for r in rows:
            h = text_hash(r["text"])
            if h in seen: continue
            seen.add(h); kept.append(r)
        if len(kept) != len(rows): log.info("content-dedup removed %d duplicate chunks", len(rows) - len(kept))
        rows = kept
    if not rows:
        log.info("all new chunks were exact duplicates of existing content — nothing to add."); return 0

    add_keywords(rows, CONFIG["keywords_top"])
    if dry_run:
        lens = [r["chunk_length"] for r in rows]
        cats = Counter(r["category"] for r in rows)
        log.info("DRY-RUN '%s': would add %d chunks | avg len %d | categories %s",
                 key, len(rows), sum(lens)//len(lens), dict(cats))
        log.info("  sample: title=%r url=%r", rows[0]["title"], rows[0].get("url"))
        return len(rows)

    # ---- embed + append ----
    if not (INDEX_FILE.exists() and META_FILE.exists()):
        raise SystemExit("Base index missing — build it first (Notebooks 1-3).")
    info = read_json(INFO_FILE, default={})
    if info.get("embedding_model") and info["embedding_model"] != CONFIG["embedding_model"]:
        raise SystemExit(f"Index built with {info['embedding_model']} but CONFIG uses "
                         f"{CONFIG['embedding_model']} — abort to avoid corruption.")
    vecs = embed_passages([r["text"] for r in rows])
    index = faiss.read_index(str(INDEX_FILE))
    if index.ntotal != len(meta): raise SystemExit(f"align mismatch: index {index.ntotal} != meta {len(meta)}")
    if vecs.shape[1] != index.d:  raise SystemExit(f"dim mismatch: {vecs.shape[1]} != {index.d}")

    backup = INDEX_DIR.parent / f"index_backup_{datetime.now():%Y%m%d_%H%M%S}"
    shutil.copytree(INDEX_DIR, backup); log.info("backed up index -> %s", backup.name)
    np.save(SHARD_DIR / f"delta_{key}_{datetime.now():%Y%m%d_%H%M%S}.npy", vecs)

    index.add(vecs); meta.extend(rows)
    if index.ntotal != len(meta): raise SystemExit("post-append alignment error — restore from backup!")
    faiss.write_index(index, str(INDEX_FILE)); write_json(META_FILE, meta)
    with CHUNKS_JSONL.open("a", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    info.update(n_vectors=int(index.ntotal), embedding_model=CONFIG["embedding_model"],
                use_e5_prefixes=True, last_updated_at=datetime.now().isoformat(timespec="seconds"))
    write_json(INFO_FILE, info)
    log.info("APPENDED %d chunks. index now %d vectors. (%.1fs)  Restart backend to load.",
             len(rows), index.ntotal, time.time() - t0)
    return len(rows)

def ingest_all(keys=None, dry_run=True):
    keys = keys or list(SOURCES)
    return {k: ingest_source(k, dry_run=dry_run) for k in keys}

## Cell 5 — Inspect: current index + which sources are already in it  (run me first)

In [ ]:
# =============================================================================
# Cell 5 — what's already indexed?
# =============================================================================
import faiss
meta = read_json(META_FILE, default=[])
ntotal = faiss.read_index(str(INDEX_FILE)).ntotal if INDEX_FILE.exists() else 0
print(f"index vectors : {ntotal:,}")
print(f"meta rows     : {len(meta):,}")
cc = Counter(r.get("category") for r in meta)
print("\nexisting categories:")
for c, n in cc.most_common():
    print(f"  {n:>8,}  {c}")
print("\nregistered sources (is the category already present?):")
for k, s in SOURCES.items():
    n = cc.get(s["category"], 0)
    print(f"  {k:<14} kind={s['kind']:<9} category={s['category']:<20} already_indexed={('YES ('+format(n,',')+')') if n else 'no'}")
print("\nNote: 'already_indexed=YES' means that category already has chunks; the ingester")
print("will still de-dupe by doc_id + exact text, so re-running won't duplicate identical content.")

## Cell 6 — DRY RUN one source (no changes)

In [ ]:
# =============================================================================
# Cell 6 — preview a source. Change KEY to any from Cell 2 (e.g. "pmb_upi").
# =============================================================================
KEY = "upi_cibiru"
_ = ingest_source(KEY, dry_run=True)

## Cell 7 — REAL INGEST (set CONFIRM=True)
Safe: append-only, de-dupes by `doc_id` + exact text, backs up the index first.
On CPU this can take a while for large sources; a GPU build of torch makes it much faster
(`CONFIG["device"]="auto"` already uses CUDA when available).

In [ ]:
# =============================================================================
# Cell 7 — embed + append for real
# =============================================================================
KEY = "upi_cibiru"        # source to ingest
CONFIRM = False           # <-- set True to actually write to the index

if CONFIRM:
    added = ingest_source(KEY, dry_run=False)
    print(f"Done — added {added} chunks for '{KEY}'.")
else:
    print("CONFIRM is False — showing dry-run preview only:\n")
    ingest_source(KEY, dry_run=True)

# To ingest several at once:
#   ingest_all(["pmb_upi", "dit_pendidikan"], dry_run=True)     # preview
#   ingest_all(["pmb_upi", "dit_pendidikan"], dry_run=False)    # commit

## Cell 8 — Retrieval sanity check

In [ ]:
# =============================================================================
# Cell 8 — quick search across the whole index
# =============================================================================
import faiss
from sentence_transformers import SentenceTransformer
_idx = faiss.read_index(str(INDEX_FILE)); _meta = read_json(META_FILE, default=[])
_m = SentenceTransformer(CONFIG["embedding_model"], device=pick_device())

def search(q, k=5):
    qv = _m.encode(["query: " + q], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    D, I = _idx.search(qv, k)
    print("Q:", q)
    for s, i in zip(D[0], I[0]):
        r = _meta[i]
        print(f"  [{s:.3f}] ({r.get('category')}) {str(r.get('title'))[:55]}  {r.get('url') or ''}")
    print()

search("visi misi UPI Kampus Cibiru")
search("program studi UPI Sumedang")
search("syarat pendaftaran mahasiswa baru UPI")